# 노드와 엣지 연결하기

이 노트북에서 다루는 연결 패턴:
1. 순차 연결
2. `add_sequence` 로 한 번에 연결
3. 병렬로 연결
4. 조건부 병렬 연결

병렬이 등장하면 여러 노드가 같은 State 필드를 동시에 갱신하므로 **리듀서가 필수**다.

## 1. 순차 연결하기

`add_node(함수)` 로 등록하면 함수명이 그대로 노드 이름이 된다.

In [ ]:
from typing_extensions import TypedDict

class State(TypedDict):
    value_1: str
    value_2: int

In [ ]:
def step_1(state: State):
    return {"value_1": state["value_1"]}

def step_2(state: State):
    current_value_1 = state["value_1"]
    return {"value_1": f"{current_value_1} b"}

def step_3(state: State):
    return {"value_2": 10}

In [ ]:
from langgraph.graph import START, END, StateGraph

graph_builder = StateGraph(State)

# 노드 추가
graph_builder.add_node(step_1)
graph_builder.add_node(step_2)
graph_builder.add_node(step_3)

# 엣지 추가
graph_builder.add_edge(START, "step_1")      # START -> 1
graph_builder.add_edge("step_1", "step_2")   # 1 -> 2
graph_builder.add_edge("step_2", "step_3")   # 2 -> 3
graph_builder.add_edge("step_3", END)        # 3 -> END

In [ ]:
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

In [ ]:
graph.invoke({"value_1": "a", "value_2": 0})

## 2. 한번에 연결하기 (`add_sequence`)

여러 노드를 차례로 잇는다면 `add_sequence([...])` 로 한 번에 표현할 수 있다. 내부 엣지(step_1 → step_2 → step_3)는 자동으로 깔리고, `START` 와 `END` 연결만 직접 해주면 된다.

In [ ]:
graph_builder = StateGraph(State).add_sequence([step_1, step_2, step_3])
graph_builder.add_edge(START, "step_1")
graph_builder.add_edge("step_3", END)

graph = graph_builder.compile()
graph.invoke({"value_1": "c", "value_2": 0})

## 3. 병렬로 연결하기

```
       ┌── b ──┐
a ─────┤        ├──> d
       └── c ──┘
```

한 노드(`a`)가 끝난 뒤 두 노드(`b`, `c`)가 **병렬 실행**되고, 둘 다 끝나면 합류 노드(`d`)가 실행된다.
여러 노드가 같은 필드(`aggregate`)를 동시에 갱신하므로 `operator.add` 리듀서로 리스트를 합친다.

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    aggregate: Annotated[list, operator.add]

In [ ]:
def a(state: State):
    print(f'Adding "A" to {state["aggregate"]}')
    return {"aggregate": ["A"]}

def b(state: State):
    print(f'Adding "B" to {state["aggregate"]}')
    return {"aggregate": ["B"]}

def c(state: State):
    print(f'Adding "C" to {state["aggregate"]}')
    return {"aggregate": ["C"]}

def d(state: State):
    print(f'Adding "D" to {state["aggregate"]}')
    return {"aggregate": ["D"]}

In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node(a)
graph_builder.add_node(b)
graph_builder.add_node(c)
graph_builder.add_node(d)

graph_builder.add_edge(START, "a")
graph_builder.add_edge("a", "b")   # a -> b
graph_builder.add_edge("a", "c")   # a -> c  (b, c 병렬)
graph_builder.add_edge("b", "d")   # b -> d
graph_builder.add_edge("c", "d")   # c -> d  (b, c 모두 끝나야 d 실행)
graph_builder.add_edge("d", END)
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

In [ ]:
graph.invoke({"aggregate": []})

## 4. 조건부 병렬 연결하기

조건부 엣지의 라우팅 함수가 **노드 이름 리스트**를 반환하면, 그 노드들이 모두 병렬 실행된다.
아래는 `which` 값에 따라 `["b", "c"]` 또는 `["c", "d"]` 를 실행하고 마지막에 `e` 로 합류한다.

In [ ]:
import operator
from typing import Annotated, Sequence
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    aggregate: Annotated[list, operator.add]
    which: str

In [ ]:
def a(state: State):
    print(f'Adding "A" to {state["aggregate"]}')
    return {"aggregate": ["A"]}

def b(state: State):
    print(f'Adding "B" to {state["aggregate"]}')
    return {"aggregate": ["B"]}

def c(state: State):
    print(f'Adding "C" to {state["aggregate"]}')
    return {"aggregate": ["C"]}

def d(state: State):
    print(f'Adding "D" to {state["aggregate"]}')
    return {"aggregate": ["D"]}

def e(state: State):
    print(f'Adding "E" to {state["aggregate"]}')
    return {"aggregate": ["E"]}

In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node(a)
graph_builder.add_node(b)
graph_builder.add_node(c)
graph_builder.add_node(d)
graph_builder.add_node(e)
graph_builder.add_edge(START, "a")

In [ ]:
# bc 혹은 cd 로 라우트를 결정하는 함수
def route_bc_or_cd(state: State) -> Sequence[str]:
    if state["which"] == "cd":
        return ["c", "d"]
    return ["b", "c"]

intermediates = ["b", "c", "d"]
graph_builder.add_conditional_edges(
    "a",
    route_bc_or_cd,
    intermediates,
)

In [ ]:
for node in intermediates:
    graph_builder.add_edge(node, "e")

graph_builder.add_edge("e", END)
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

In [ ]:
graph.invoke({"aggregate": [], "which": "bc"})

In [ ]:
graph.invoke({"aggregate": [], "which": "cd"})

## 정리

| 패턴 | 핵심 |
|---|---|
| 순차 | `add_edge(src, dst)` 반복 |
| 순차(단축) | `add_sequence([fn1, fn2, ...])` |
| 병렬 | 같은 src 에서 여러 dst 로 `add_edge` |
| 조건부 병렬 | 라우터가 `list[str]` 반환 + `add_conditional_edges(src, router, [...])` |

다음: 조건에 따른 반복(루프)과 동적 라우팅을 다룬다.